## Extract Flats 

In [6]:
import os
import pandas as pd
from bs4 import BeautifulSoup

# Folder containing all HTML files
folder_path = 'C:/Users/91892/Documents/#MLpractice/real_estate/full_flats'

# Store results
data = []

# Loop through all HTML files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".html"):
        with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
            html = f.read()

        soup = BeautifulSoup(html, 'html.parser')
        result = {}

        # Property name
        try:
            result['property_name'] = soup.select_one("div.tupleNew__locationName.ellipsis").text.strip()
        except:
            result['property_name'] = ''

        # Link
        try:
            link_tag = soup.select_one("a.tupleNew__propertyHeading.ellipsis")
            result['link'] = link_tag['href'] if link_tag else ''
        except:
            result['link'] = ''

        # Location
        try:
            result['location'] = soup.select_one("a.tupleNew__propertyHeading.ellipsis").text.strip()
        except:
            result['location'] = ''

        # Price
        try:
            price_tag = soup.select_one("div.tupleNew__priceValWrap span")
            result['price'] = price_tag.get_text(strip=True) if price_tag else ''
        except:
            result['price'] = ''

        # Price per sqft
        try:
            ppsf_tag = soup.select_one("div.tupleNew__perSqftWrap.ellipsis")
            result['price_per_sqft'] = ppsf_tag.get_text(strip=True) if ppsf_tag else ''
        except:
            result['price_per_sqft'] = ''

        # Area
        # try:
        #     area_tag = soup.select_one("span.tupleNew__area1Type")
        #     result['area'] = area_tag.get_text(strip=True) if area_tag else ''
        # except:
        #     result['area'] = ''

        # # Area type
        # try:
        #     area_type_tag = soup.select_one("div.tupleNew__areaType")
        #     result['area_type'] = area_type_tag.get_text(strip=True) if area_type_tag else ''
        # except:
        #     result['area_type'] = ''

        # Super built-up area
        try:
            sba_tag = soup.select_one("span#superbuiltupArea_span")
            result['super_built_up_area'] = int(sba_tag.get_text(strip=True)) if sba_tag else None
        except:
            result['super_built_up_area'] = None

        # Carpet area
        try:
            carpet_tag = soup.select_one("span#carpetArea_span")
            result['carpet_area'] = int(carpet_tag.get_text(strip=True)) if carpet_tag else None
        except:
            result['carpet_area'] = None

        # Built-up area
        try:
            ba=soup.select_one('span#builtupArea_span')
            result['builtup_area'] = int(ba.get_text(strip=True)) if ba else None
        except:
            result['builtup_area'] = None


        # Description
        try:
            desc_tag = soup.select_one("p.tupleNew__descText")
            result['description'] = desc_tag.get_text(strip=True) if desc_tag else ''
        except:
            result['description'] = ''

        # Facing
        try:
            facing_tag = soup.select_one("span#facingLabel")
            result['facing'] = facing_tag.get_text(strip=True) if facing_tag else ''
        except:
            result['facing'] = ''

        # Highlights
        try:
            highlights = soup.select("div.tupleNew__unitHighlightTxt, div.tupleNew__unitHighlightTxtLast")
            result['highlights'] = [h.get_text(strip=True) for h in highlights]
        except:
            result['highlights'] = []

        # Fallback Features
        fallback_items = soup.select("#amenitiesWrp div#xidAmeWrap div.undefined")
        result['features_list'] = [
            div.get_text(strip=True)
            for div in fallback_items
            if div.get_text(strip=True)
        ]

        # Property ID
        try:
            prop_id_tag = soup.select_one("span#Prop_Id")
            result['property_id'] = prop_id_tag.get_text(strip=True) if prop_id_tag else ''
        except:
            result['property_id'] = ''

        # Ownership type
        try:
            owner_tag = soup.select_one("span#Owntype_Label")
            result['ownership'] = owner_tag.get_text(strip=True) if owner_tag else ''
        except:
            result['ownership'] = ''

        # Gated community
        try:
            gated_tag = soup.select_one("span#Gated_community")
            result['gated_community'] = gated_tag.get_text(strip=True) if gated_tag else ''
        except:
            result['gated_community'] = ''

        # Bedroom / Bathroom / Balcony / Others
        for key, selector in {
            'bedrooms': '#bedRoomNum',
            'bathrooms': '#bathroomNum',
            'balcony': '#balconyNum',
            'others': '#additionalRooms',
            'floor': '#floorNumLabel',
            'overlooking': '#overlooking',
            'age': '#agePossessionLbl',
            'furnish_type': '#Furnish_Label'
        }.items():
            try:
                result[key] = soup.select_one(selector).text.strip()
            except:
                result[key] = ''

        # Nearby locations
        try:
            result['nearby_locations_list'] = [tag.text.strip() for tag in soup.select("span.NearByLocation__infoText")]
        except:
            result['nearby_locations_list'] = []

        # Overall rating
        try:
            rating_div = soup.select_one("div.rc__ratingWrap div.display_l_semiBold")
            result['rating'] = rating_div.text.strip() if rating_div else ''
        except:
            result['rating'] = ''

        # Rating details by feature
        try:
            result['rating_details'] = {}
            rating_blocks = soup.select(".ratingByFeature__circleWrap")
            if rating_blocks:
                for block in rating_blocks:
                    feature_tag = block.select_one(".list_header_semiBold")
                    rating_tag = block.select_one(".caption_subdued_medium")
                    if feature_tag and rating_tag:
                        feature = feature_tag.text.strip()
                        rating_text = rating_tag.text.strip()
                        rating_value = float(rating_text.split()[0])
                        result['rating_details'][feature] = rating_value
        except:
            result['rating_details'] = {}

        # Furnish details
        try:
            result['furnish_details_list'] = [
                div.get_text(strip=True)
                for div in soup.select('#FurnishDetails ul li div')
            ]
        except:
            result['furnish_details_list'] = []

        # Extract features (excluding those in FurnishDetails)
        try:
            result['features'] = []
            feature_blocks = soup.select("div.component__features.pd__pdBlock")
            for block in feature_blocks:
                if block.find_parent(id="FurnishDetails"):
                    continue
                heading = block.find("h2")
                if heading and "feature" in heading.text.lower():
                    items = block.select("ul li div")
                    result['features'].extend(div.get_text(strip=True) for div in items)
        except:
            result['features'] = []

        # Append result for this file
        data.append(result)

# Create DataFrame
new_df = pd.DataFrame(data)

# Preview
print(new_df.head())


         property_name                                               link  \
0         Lodha Allura  https://www.99acres.com/3-bhk-bedroom-apartmen...   
1    DSS Mahavir Meena  https://www.99acres.com/3-bhk-bedroom-apartmen...   
2  Agarwal Sky Heights  https://www.99acres.com/1-bhk-bedroom-apartmen...   
3     Oberoi Exquisite  https://www.99acres.com/3-bhk-bedroom-apartmen...   
4        VASWANI VISTA  https://www.99acres.com/2-bhk-bedroom-apartmen...   

                               location                price price_per_sqft  \
0     3 BHK Flat in Lower Parel, Mumbai              ₹6.9 Cr  ₹63,186 /sqft   
1  3 BHK Flat in Ghatkopar East, Mumbai              ₹4.5 Cr  ₹38,986 /sqft   
2      1 BHK Flat in Vasai East, Mumbai  ₹46.98  - 49.79 Lac  ₹11,046 /sqft   
3   3 BHK Flat in Goregaon East, Mumbai              ₹7.3 Cr  ₹40,109 /sqft   
4  2 BHK Flat in Kandivali West, Mumbai             ₹1.56 Cr  ₹32,098 /sqft   

   super_built_up_area  carpet_area  builtup_area  \
0        

In [7]:
new_df.shape

(5403, 28)

In [8]:
# Step 1: Load existing file (if exists)
try:
    existing_df = pd.read_csv("flats_data.csv")
except FileNotFoundError:
    existing_df = pd.DataFrame(columns=new_df.columns)

# Step 2: Combine
combined_df = pd.concat([existing_df, new_df])

combined_df.to_csv("flats_data.csv", index=False)

# Done
print("✅ Data appended ")
print("📊 Final shape:", combined_df.shape)

✅ Data appended 
📊 Final shape: (5403, 28)


## Extract Houses

In [9]:
import os
import pandas as pd
from bs4 import BeautifulSoup

# Folder containing all HTML files
folder_path = 'C:/Users/91892/Documents/#MLpractice/real_estate/full_houses'

# Store results
data = []

# Loop through all HTML files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".html"):
        with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
            html = f.read()

        soup = BeautifulSoup(html, 'html.parser')
        result = {}

        # Property name
        try:
            result['property_name'] = soup.select_one("div.tupleNew__locationName.ellipsis").text.strip()
        except:
            result['property_name'] = ''

        # Link
        try:
            link_tag = soup.select_one("a.tupleNew__propertyHeading.ellipsis")
            result['link'] = link_tag['href'] if link_tag else ''
        except:
            result['link'] = ''

        # Location
        try:
            result['location'] = soup.select_one("a.tupleNew__propertyHeading.ellipsis").text.strip()
        except:
            result['location'] = ''

        # Price
        try:
            price_tag = soup.select_one("div.tupleNew__priceValWrap span")
            result['price'] = price_tag.get_text(strip=True) if price_tag else ''
        except:
            result['price'] = ''

        # Price per sqft
        try:
            ppsf_tag = soup.select_one("div.tupleNew__perSqftWrap.ellipsis")
            result['price_per_sqft'] = ppsf_tag.get_text(strip=True) if ppsf_tag else ''
        except:
            result['price_per_sqft'] = ''

        # Area
        # try:
        #     area_tag = soup.select_one("span.tupleNew__area1Type")
        #     result['area'] = area_tag.get_text(strip=True) if area_tag else ''
        # except:
        #     result['area'] = ''

        # # Area type
        # try:
        #     area_type_tag = soup.select_one("div.tupleNew__areaType")
        #     result['area_type'] = area_type_tag.get_text(strip=True) if area_type_tag else ''
        # except:
        #     result['area_type'] = ''

        try:
            sba_tag = soup.select_one("span#superbuiltupArea_span")
            result['super_built_up_area'] = int(sba_tag.get_text(strip=True)) if sba_tag else None
        except:
            result['super_built_up_area'] = None

        # Carpet area
        try:
            carpet_tag = soup.select_one("span#carpetArea_span")
            result['carpet_area'] = int(carpet_tag.get_text(strip=True)) if carpet_tag else None
        except:
            result['carpet_area'] = None

        # Built-up area
        try:
            ba=soup.select_one('span#builtupArea_span')
            result['builtup_area'] = int(ba.get_text(strip=True)) if ba else None
        except:
            result['builtup_area'] = None

        # Description
        try:
            desc_tag = soup.select_one("p.tupleNew__descText")
            result['description'] = desc_tag.get_text(strip=True) if desc_tag else ''
        except:
            result['description'] = ''

        # Facing
        try:
            facing_tag = soup.select_one("span#facingLabel")
            result['facing'] = facing_tag.get_text(strip=True) if facing_tag else ''
        except:
            result['facing'] = ''

        # Highlights
        try:
            highlights = soup.select("div.tupleNew__unitHighlightTxt, div.tupleNew__unitHighlightTxtLast")
            result['highlights'] = [h.get_text(strip=True) for h in highlights]
        except:
            result['highlights'] = []

        # Fallback Features
        fallback_items = soup.select("#amenitiesWrp div#xidAmeWrap div.undefined")
        result['features_list'] = [
            div.get_text(strip=True)
            for div in fallback_items
            if div.get_text(strip=True)
        ]

        # Property ID
        try:
            prop_id_tag = soup.select_one("span#Prop_Id")
            result['property_id'] = prop_id_tag.get_text(strip=True) if prop_id_tag else ''
        except:
            result['property_id'] = ''

        # Ownership type
        try:
            owner_tag = soup.select_one("span#Owntype_Label")
            result['ownership'] = owner_tag.get_text(strip=True) if owner_tag else ''
        except:
            result['ownership'] = ''

        # Gated community
        try:
            gated_tag = soup.select_one("span#Gated_community")
            result['gated_community'] = gated_tag.get_text(strip=True) if gated_tag else ''
        except:
            result['gated_community'] = ''

        # Bedroom / Bathroom / Balcony / Others
        for key, selector in {
            'bedrooms': '#bedRoomNum',
            'bathrooms': '#bathroomNum',
            'balcony': '#balconyNum',
            'others': '#additionalRooms',
            'floor': '#floorNumLabel',
            'overlooking': '#overlooking',
            'age': '#agePossessionLbl',
            'furnish_type': '#Furnish_Label'
        }.items():
            try:
                result[key] = soup.select_one(selector).text.strip()
            except:
                result[key] = ''

        # Nearby locations
        try:
            result['nearby_locations_list'] = [tag.text.strip() for tag in soup.select("span.NearByLocation__infoText")]
        except:
            result['nearby_locations_list'] = []

        # Overall rating
        try:
            rating_div = soup.select_one("div.rc__ratingWrap div.display_l_semiBold")
            result['rating'] = rating_div.text.strip() if rating_div else ''
        except:
            result['rating'] = ''

        # Rating details by feature
        try:
            result['rating_details'] = {}
            rating_blocks = soup.select(".ratingByFeature__circleWrap")
            if rating_blocks:
                for block in rating_blocks:
                    feature_tag = block.select_one(".list_header_semiBold")
                    rating_tag = block.select_one(".caption_subdued_medium")
                    if feature_tag and rating_tag:
                        feature = feature_tag.text.strip()
                        rating_text = rating_tag.text.strip()
                        rating_value = float(rating_text.split()[0])
                        result['rating_details'][feature] = rating_value
        except:
            result['rating_details'] = {}

        # Furnish details
        try:
            result['furnish_details_list'] = [
                div.get_text(strip=True)
                for div in soup.select('#FurnishDetails ul li div')
            ]
        except:
            result['furnish_details_list'] = []

        # Extract features (excluding those in FurnishDetails)
        try:
            result['features'] = []
            feature_blocks = soup.select("div.component__features.pd__pdBlock")
            for block in feature_blocks:
                if block.find_parent(id="FurnishDetails"):
                    continue
                heading = block.find("h2")
                if heading and "feature" in heading.text.lower():
                    items = block.select("ul li div")
                    result['features'].extend(div.get_text(strip=True) for div in items)
        except:
            result['features'] = []

        # Append result for this file
        data.append(result)

# Create DataFrame
new_df = pd.DataFrame(data)

# Preview
print(new_df.head())


                             property_name  \
0  Charkop, Kandivali West, Western Mumbai   
1   Kandivali West, Mumbai, Western Mumbai   
2                             Reay Chamber   
3           Karjat, Thane, Thane Outskirts   
4           Karjat, Thane, Thane Outskirts   

                                                link  \
0  https://www.99acres.com/4-bhk-bedroom-independ...   
1  https://www.99acres.com/6-bhk-bedroom-independ...   
2  https://www.99acres.com/1-bhk-bedroom-independ...   
3  https://www.99acres.com/2-bhk-bedroom-independ...   
4  https://www.99acres.com/3-bhk-bedroom-independ...   

                                            location    price price_per_sqft  \
0  4 Bedroom House for sale in Charkop, Kandivali...    ₹4 Cr  ₹22,222 /sqft   
1  6 Bedroom House for sale in Kandivali West, Mu...   ₹18 Cr  ₹40,000 /sqft   
2        1 Bedroom House for sale in Mazgaon, Mumbai  ₹60 Lac  ₹20,000 /sqft   
3          2 Bedroom House for sale in Karjat, Thane  ₹98 Lac  ₹10

In [10]:
new_df.shape

(1356, 28)

In [11]:
# Step 1: Load existing file (if exists)
try:
    existing_df = pd.read_csv("houses_data.csv")
except FileNotFoundError:
    existing_df = pd.DataFrame(columns=new_df.columns)

# Step 2: Combine
combined_df = pd.concat([existing_df, new_df])

combined_df.to_csv("houses_data.csv", index=False)

# Done
print("✅ Data appended ")
print("📊 Final shape:", combined_df.shape)

✅ Data appended 
📊 Final shape: (1356, 28)


## Extract Resale

In [12]:
import os
import pandas as pd
from bs4 import BeautifulSoup

# Folder containing all HTML files
folder_path = 'C:/Users/91892/Documents/#MLpractice/real_estate/full_resales'

# Store results
data = []

# Loop through all HTML files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".html"):
        with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
            html = f.read()

        soup = BeautifulSoup(html, 'html.parser')
        result = {}

        # Property name
        try:
            result['property_name'] = soup.select_one("div.tupleNew__locationName.ellipsis").text.strip()
        except:
            result['property_name'] = ''

        # Link
        try:
            link_tag = soup.select_one("a.tupleNew__propertyHeading.ellipsis")
            result['link'] = link_tag['href'] if link_tag else ''
        except:
            result['link'] = ''

        # Location
        try:
            result['location'] = soup.select_one("a.tupleNew__propertyHeading.ellipsis").text.strip()
        except:
            result['location'] = ''

        # Price
        try:
            price_tag = soup.select_one("div.tupleNew__priceValWrap span")
            result['price'] = price_tag.get_text(strip=True) if price_tag else ''
        except:
            result['price'] = ''

        # Price per sqft
        try:
            ppsf_tag = soup.select_one("div.tupleNew__perSqftWrap.ellipsis")
            result['price_per_sqft'] = ppsf_tag.get_text(strip=True) if ppsf_tag else ''
        except:
            result['price_per_sqft'] = ''

        # Area
        # try:
        #     area_tag = soup.select_one("span.tupleNew__area1Type")
        #     result['area'] = area_tag.get_text(strip=True) if area_tag else ''
        # except:
        #     result['area'] = ''

        # # Area type
        # try:
        #     area_type_tag = soup.select_one("div.tupleNew__areaType")
        #     result['area_type'] = area_type_tag.get_text(strip=True) if area_type_tag else ''
        # except:
        #     result['area_type'] = ''

        try:
            sba_tag = soup.select_one("span#superbuiltupArea_span")
            result['super_built_up_area'] = int(sba_tag.get_text(strip=True)) if sba_tag else None
        except:
            result['super_built_up_area'] = None

        # Carpet area
        try:
            carpet_tag = soup.select_one("span#carpetArea_span")
            result['carpet_area'] = int(carpet_tag.get_text(strip=True)) if carpet_tag else None
        except:
            result['carpet_area'] = None

        # Built-up area
        try:
            ba=soup.select_one('span#builtupArea_span')
            result['builtup_area'] = int(ba.get_text(strip=True)) if ba else None
        except:
            result['builtup_area'] = None

        # Description
        try:
            desc_tag = soup.select_one("p.tupleNew__descText")
            result['description'] = desc_tag.get_text(strip=True) if desc_tag else ''
        except:
            result['description'] = ''

        # Facing
        try:
            facing_tag = soup.select_one("span#facingLabel")
            result['facing'] = facing_tag.get_text(strip=True) if facing_tag else ''
        except:
            result['facing'] = ''

        # Highlights
        try:
            highlights = soup.select("div.tupleNew__unitHighlightTxt, div.tupleNew__unitHighlightTxtLast")
            result['highlights'] = [h.get_text(strip=True) for h in highlights]
        except:
            result['highlights'] = []

        # Fallback Features
        fallback_items = soup.select("#amenitiesWrp div#xidAmeWrap div.undefined")
        result['features_list'] = [
            div.get_text(strip=True)
            for div in fallback_items
            if div.get_text(strip=True)
        ]

        # Property ID
        try:
            prop_id_tag = soup.select_one("span#Prop_Id")
            result['property_id'] = prop_id_tag.get_text(strip=True) if prop_id_tag else ''
        except:
            result['property_id'] = ''

        # Ownership type
        try:
            owner_tag = soup.select_one("span#Owntype_Label")
            result['ownership'] = owner_tag.get_text(strip=True) if owner_tag else ''
        except:
            result['ownership'] = ''

        # Gated community
        try:
            gated_tag = soup.select_one("span#Gated_community")
            result['gated_community'] = gated_tag.get_text(strip=True) if gated_tag else ''
        except:
            result['gated_community'] = ''

        # Bedroom / Bathroom / Balcony / Others
        for key, selector in {
            'bedrooms': '#bedRoomNum',
            'bathrooms': '#bathroomNum',
            'balcony': '#balconyNum',
            'others': '#additionalRooms',
            'floor': '#floorNumLabel',
            'overlooking': '#overlooking',
            'age': '#agePossessionLbl',
            'furnish_type': '#Furnish_Label'
        }.items():
            try:
                result[key] = soup.select_one(selector).text.strip()
            except:
                result[key] = ''

        # Nearby locations
        try:
            result['nearby_locations_list'] = [tag.text.strip() for tag in soup.select("span.NearByLocation__infoText")]
        except:
            result['nearby_locations_list'] = []

        # Overall rating
        try:
            rating_div = soup.select_one("div.rc__ratingWrap div.display_l_semiBold")
            result['rating'] = rating_div.text.strip() if rating_div else ''
        except:
            result['rating'] = ''

        # Rating details by feature
        try:
            result['rating_details'] = {}
            rating_blocks = soup.select(".ratingByFeature__circleWrap")
            if rating_blocks:
                for block in rating_blocks:
                    feature_tag = block.select_one(".list_header_semiBold")
                    rating_tag = block.select_one(".caption_subdued_medium")
                    if feature_tag and rating_tag:
                        feature = feature_tag.text.strip()
                        rating_text = rating_tag.text.strip()
                        rating_value = float(rating_text.split()[0])
                        result['rating_details'][feature] = rating_value
        except:
            result['rating_details'] = {}

        # Furnish details
        try:
            result['furnish_details_list'] = [
                div.get_text(strip=True)
                for div in soup.select('#FurnishDetails ul li div')
            ]
        except:
            result['furnish_details_list'] = []

        # Extract features (excluding those in FurnishDetails)
        try:
            result['features'] = []
            feature_blocks = soup.select("div.component__features.pd__pdBlock")
            for block in feature_blocks:
                if block.find_parent(id="FurnishDetails"):
                    continue
                heading = block.find("h2")
                if heading and "feature" in heading.text.lower():
                    items = block.select("ul li div")
                    result['features'].extend(div.get_text(strip=True) for div in items)
        except:
            result['features'] = []

        # Append result for this file
        data.append(result)

# Create DataFrame
new_df = pd.DataFrame(data)

# Preview
print(new_df.head())


       property_name                                               link  \
0   Shreeji Atlantis  https://www.99acres.com/2-bhk-bedroom-apartmen...   
1  Mavani Geetanjali  https://www.99acres.com/3-bhk-bedroom-apartmen...   
2   Mhada Lig Colony  https://www.99acres.com/1-bhk-bedroom-apartmen...   
3      Ishwar Acacia  https://www.99acres.com/2-bhk-bedroom-apartmen...   
4  Sadguru Universal  https://www.99acres.com/2-bhk-bedroom-apartmen...   

                                    location    price price_per_sqft  \
0           2 BHK Flat in Malad West, Mumbai  ₹2.2 Cr  ₹29,972 /sqft   
1       3 BHK Flat in Ghatkopar East, Mumbai  ₹2.8 Cr  ₹29,442 /sqft   
2           1 BHK Flat in Kurla West, Mumbai  ₹65 Lac  ₹24,163 /sqft   
3  2 BHK Flat in Sector 17 Ulwe, Navi Mumbai  ₹99 Lac  ₹14,142 /sqft   
4        2 BHK Flat in Khanda Colony, Panvel  ₹1.2 Cr  ₹19,200 /sqft   

   super_built_up_area  carpet_area  builtup_area  \
0                  NaN        734.0           NaN   
1         

In [13]:
new_df.shape

(3639, 28)

In [14]:
# Step 1: Load existing file (if exists)
try:
    existing_df = pd.read_csv("resales_data.csv")
except FileNotFoundError:
    existing_df = pd.DataFrame(columns=new_df.columns)

# Step 2: Combine
combined_df = pd.concat([existing_df, new_df])

combined_df.to_csv("resales_data.csv", index=False)

# Done
print("✅ Data appended ")
print("📊 Final shape:", combined_df.shape)

✅ Data appended 
📊 Final shape: (3639, 28)


In [15]:
# import os
# import shutil

# source_folder = "C:/Users/91892/Documents/#Programming/#Project/python/scrape/data_flats"     # Folder containing subfolders
# destination_folder = "C:/Users/91892/Documents/#Programming/#Project/python/scrape/data_flats"       # Where you want all files combined

# # Make sure the destination folder exists
# os.makedirs(destination_folder, exist_ok=True)

# counter = 3633  # To give unique sequential names

# # Walk through all subfolders and collect HTML files
# for root, dirs, files in os.walk(source_folder):
#     for file in files:
#         if file.endswith(".html"):
#             # Build new filename with continuous numbering
#             new_name = f"flat_{counter}.html"
#             counter += 1

#             # Copy file to destination folder with new name
#             src_path = os.path.join(root, file)
#             dst_path = os.path.join(destination_folder, new_name)
#             shutil.copy2(src_path, dst_path)

# print(f"All files moved and renamed. Total files: {counter-1}")
